In [17]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier


In [18]:
df = pd.read_csv("/home/shaiv/Documents/logestic reg/user-analytics-ml-system/data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [19]:
df = df.dropna(subset=["TotalCharges"])

In [20]:
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

In [21]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [23]:
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = X.select_dtypes(include=["object"]).columns

/tmp/ipykernel_405964/320056118.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object"]).columns


In [24]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(drop="first",handle_unknown="ignore"), categorical_cols)
    ]
)

In [25]:
xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=2.8,
    random_state=42
)

In [26]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", xgb_model)
])


In [27]:
scale_pos_weight = 2.8

In [28]:
param_grid = {

    "model__n_estimators": [200, 400, 600],

    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],

    "model__max_depth": [3, 4, 5, 6, 8],

    "model__min_child_weight": [1, 3, 5],

    "model__subsample": [0.6, 0.8, 1.0],

    "model__colsample_bytree": [0.6, 0.8, 1.0],

    "model__gamma": [0, 0.1, 0.3, 0.5],

    "model__reg_alpha": [0, 0.1, 1],

    "model__reg_lambda": [1, 3, 5]
}

In [29]:
random_search = RandomizedSearchCV(

    estimator=pipeline,

    param_distributions=param_grid,

    n_iter=25,

    scoring="f1",

    cv=5,

    verbose=2,

    n_jobs=-1,

    random_state=42
)

# Train

random_search.fit(X_train, y_train)

# Best parameters

print("Best Parameters:")
print(random_search.best_params_)

# Best score

print("Best F1 Score:")
print(random_search.best_score_)

# Best model

best_model = random_search.best_estimator_

# Predictions

y_pred = best_model.predict(X_test)

# Report

print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 25 candidates, totalling 125 fits


/home/shaiv/Documents/logestic reg/user-analytics-ml-system/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [0, 16] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/shaiv/Documents/logestic reg/user-analytics-ml-system/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [0, 16] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/shaiv/Documents/logestic reg/user-analytics-ml-system/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [0, 16] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/shaiv/Documents/logestic reg/user-analytics-ml-system/.venv/lib/python3.13/site-packages/sklearn/pre

[CV] END model__colsample_bytree=0.6, model__gamma=0.5, model__learning_rate=0.03, model__max_depth=3, model__min_child_weight=1, model__n_estimators=200, model__reg_alpha=0, model__reg_lambda=1, model__subsample=0.6; total time=   1.2s
[CV] END model__colsample_bytree=0.6, model__gamma=0.5, model__learning_rate=0.03, model__max_depth=3, model__min_child_weight=1, model__n_estimators=200, model__reg_alpha=0, model__reg_lambda=1, model__subsample=0.6; total time=   1.2s
[CV] END model__colsample_bytree=0.6, model__gamma=0.5, model__learning_rate=0.03, model__max_depth=3, model__min_child_weight=1, model__n_estimators=200, model__reg_alpha=0, model__reg_lambda=1, model__subsample=0.6; total time=   1.2s
[CV] END model__colsample_bytree=0.6, model__gamma=0.5, model__learning_rate=0.03, model__max_depth=3, model__min_child_weight=1, model__n_estimators=200, model__reg_alpha=0, model__reg_lambda=1, model__subsample=0.6; total time=   1.2s
[CV] END model__colsample_bytree=0.6, model__gamma=0

/home/shaiv/Documents/logestic reg/user-analytics-ml-system/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [0, 16] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


[CV] END model__colsample_bytree=0.8, model__gamma=0.5, model__learning_rate=0.1, model__max_depth=5, model__min_child_weight=1, model__n_estimators=200, model__reg_alpha=0, model__reg_lambda=5, model__subsample=0.8; total time=   3.8s


...

/home/shaiv/Documents/logestic reg/user-analytics-ml-system/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [0, 16] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/shaiv/Documents/logestic reg/user-analytics-ml-system/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [0, 16] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


[CV] END model__colsample_bytree=0.6, model__gamma=0.5, model__learning_rate=0.05, model__max_depth=5, model__min_child_weight=1, model__n_estimators=600, model__reg_alpha=1, model__reg_lambda=1, model__subsample=0.6; total time=   7.2s
[CV] END model__colsample_bytree=0.6, model__gamma=0.5, model__learning_rate=0.05, model__max_depth=5, model__min_child_weight=1, model__n_estimators=600, model__reg_alpha=1, model__reg_lambda=1, model__subsample=0.6; total time=   7.5s
[CV] END model__colsample_bytree=0.6, model__gamma=0.5, model__learning_rate=0.05, model__max_depth=5, model__min_child_weight=1, model__n_estimators=600, model__reg_alpha=1, model__reg_lambda=1, model__subsample=0.6; total time=   7.3s


/home/shaiv/Documents/logestic reg/user-analytics-ml-system/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [0, 16] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Best Parameters:
{'model__subsample': 0.8, 'model__reg_lambda': 5, 'model__reg_alpha': 1, 'model__n_estimators': 200, 'model__min_child_weight': 3, 'model__max_depth': 4, 'model__learning_rate': 0.03, 'model__gamma': 0.3, 'model__colsample_bytree': 0.6}
Best F1 Score:
0.6283712134657027
              precision    recall  f1-score   support

           0       0.93      0.73      0.81      1036
           1       0.53      0.84      0.65       373

    accuracy                           0.76      1409
   macro avg       0.73      0.78      0.73      1409
weighted avg       0.82      0.76      0.77      1409



/home/shaiv/Documents/logestic reg/user-analytics-ml-system/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [0, 16] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
